# oold AI playground

An LLM agent in the left sidebar and a Pyodide kernel behind it. The agent runs Python
through the `jupyterlab-ai-commands:execute-in-kernel` command, which uses the kernel of
this notebook, so everything it does lands here and you can inspect it from these cells.

Nothing leaves the browser except the calls to your model provider. There is no server:
the kernel, the files and the schemas live on the contents drive at `/drive`.

## Before the first message

Open **AI Settings** from the chat toolbar and add a provider with its API key. That is
the only step.

The system prompt is already filled in. The site ships `OOLD-AGENT.md` as the default, so
the agent arrives knowing the Python API, what a check id means, and that this kernel has
no shell. The file sits next to this notebook; edit the prompt in **AI Settings** to change
what the agent is told.

## Bootstrap

Two lines. `oold_agent` has to be imported before anything touches `oold`, because it sets
`OOLD_DESCRIPTOR_BINDING` at import time. `setup` installs the wheels into the kernel and is
the only coroutine in the module.

In [ ]:
import oold_agent

await oold_agent.setup()
print("AI PLAYGROUND READY")

## First instruction

Send this to the agent:

> Profile `sample.csv`, draft an OO-LD schema for it, validate the draft before you
> write it to disk, and report the check id of anything that fails.

Spreadsheets work the same way. `xlsx_profile(path, sheet=None)` returns what
`csv_profile` returns plus `sheet` and `sheets`, and `sheets(path)` lists the sheet names
on their own.

Documents too. `doc_profile(path, table=None)` profiles one table out of a docx or a
pptx into that same column shape, `tables(path)` lists the tables, and `doc_text(path)`
returns the prose around them. A pdf has no table extraction, so read it with
`doc_text` and take the columns off the page text.

There is one CSV on the drive to start from. Ask the agent to pull a repository in when
you want more:

> Pull the CSVs from SINTEF/physmet-data-documentation-templates and profile one.


In [ ]:
profile = oold_agent.csv_profile("sample.csv")
print(profile["rows"], "rows,", len(profile["columns"]), "columns")
for column in profile["columns"]:
    print(column["name"], column["type"], column["examples"])


## GitHub

`oold_git` reaches the GitHub REST API from the kernel. Reads need no token, writes need a
fine-grained PAT. Always pass `paths`: without it the whole tree is fetched at one request
per file and unauthenticated GitHub allows 60 per hour.

```python
import piplite
await piplite.install(["requests", "pyodide-http"])
import oold_git

oold_git.pull("SINTEF/physmet-data-documentation-templates", into="gh",
              paths=["shared/*.csv", "context/context.json"])
```